In [ ]:
import numpy as np
import scipy.io

mat = scipy.io.loadmat("Example_Data.mat", simplify_cells=True)
print("Variables found:", [k for k in mat.keys() if not k.startswith("__")])

Variables found: ['X1', 'X2', 'X3', 'X4', 'dt1', 'times']


In [ ]:
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

In [ ]:
mat

{'__header__': b'MATLAB 5.0 MAT-file, Platform: MACI64, Created on: Thu Sep 19 14:47:05 2019',
 '__version__': '1.0',
 '__globals__': [],
 'X1': array([[ 3.42908546,  3.19274123,  3.53839328,  3.31847314,  3.69705432,  2.82817993,  3.01008433,  2.92454082,  2.71791902,  3.18131067,  2.88045811,  2.34866792,  2.73032284,  2.58692595,  2.46522401,  2.98124499,  3.16088835,  3.58536949,  2.99753553,  2.24732407,  4.41721861],
        [ 1.41638522,  3.0676972 ,  3.1863934 ,  2.86228189,  3.02973054,  3.17748439,  4.07581682,  3.51979283,  2.70618893,  2.64616315,  3.37179791,  4.10344852,  3.65476354,  3.7197787 ,  2.979342  ,  3.25321735,  4.65458835,  4.62677458,  4.23685337,  4.27564477,  4.04994982],
        [10.47417432,  5.64088488,  4.27369217,  3.17409743,  2.35662506,  3.2554804 ,  3.41231532,  3.16116015,  2.62988054,  2.16487516,  3.12122632,  3.36231078,  4.39396574,  3.32801989,  3.55258272,  2.99177057,  3.70009994,  4.66984447,  4.72019031,  4.15290949,  4.3785778 ],
       

In [ ]:

import numpy as np
from dataclasses import dataclass
from typing import Optional
from helpers.BINGOdata import BINGOData, BINGOParams, BINGOState

In [ ]:
params = BINGOParams(nstep=4, its=1)
data = BINGOData(ts=[mat['X1'], mat['X2']], Tsam=[mat["dt1"], mat['times']], params=params)

In [18]:

def bingo_init(data: BINGOData, params: BINGOParams, scale: bool = True) -> tuple[BINGOData, BINGOState]:
    """
    Initialises BINGO. If scale=True, each gene is scaled so that its
    range across all experiments maps to [0, 1]. Modifies data.ts in place.
    Returns (data, state).
    """
    # ── Scaling ────────────────────────────────────────────────────────────────
    if scale:
        maxs = np.full(data.n_genes, -np.inf)
        mins = np.full(data.n_genes,  np.inf)
        for ts_i in data.ts:
            maxs = np.maximum(maxs, ts_i.max(axis=1))
            mins = np.minimum(mins, ts_i.min(axis=1))
        scale_factor = maxs - mins          # [n_genes]
        data.ts = [ts_i / scale_factor[:, None] for ts_i in data.ts]
        # recompute maxs/mins in scaled space
        maxs = maxs / scale_factor
        mins = maxs - 1.0
    else:
        maxs = np.full(data.n_genes, 1.0)
        mins = np.zeros(data.n_genes)

    n      = data.n_genes
    n_in   = data.input[0].shape[0] if data.input is not None else 0
    nstep  = data.params.nstep
    nr_pi  = data.params.nr_pi
    n_exp  = data.n_experiments

    # ── Ser matrix [4 x n_experiments] ────────────────────────────────────────
    # Rows 0-1: start/end indices in the coarse (measurement) grid (0-based)
    # Rows 2-3: start/end indices in the fine (interpolated) grid (0-based)
    Ser = np.zeros((4, n_exp), dtype=int)
    Ser[0, 0] = 0
    Ser[1, 0] = data.ts[0].shape[1] - 1
    Ser[2, 0] = 0
    Ser[3, 0] = nstep * (Ser[1, 0] - Ser[0, 0])
    for j in range(1, n_exp):
        n_tp_j = data.ts[j].shape[1]
        Ser[0, j] = Ser[1, j-1] + 1
        Ser[1, j] = Ser[0, j] + n_tp_j - 1
        Ser[2, j] = Ser[3, j-1] + 1
        Ser[3, j] = Ser[2, j] + nstep * (n_tp_j - 1)

    # ── Initial trajectory: linear interpolation between measurements ──────────
    total_fine = Ser[3, -1] + 1
    xs = np.zeros((n, total_fine))
    for j in range(n_exp):
        ts_j = data.ts[j]
        n_tp = ts_j.shape[1]
        for jj in range(n_tp - 1):
            t = np.arange(nstep) / nstep           # [0, 1/nstep, ..., (nstep-1)/nstep]
            col_start = Ser[2, j] + jj * nstep
            xs[:, col_start:col_start + nstep] = (
                ts_j[:, jj:jj+1] * (1 - t) + ts_j[:, jj+1:jj+2] * t
            )
        xs[:, Ser[3, j]] = ts_j[:, -1]

    # ── Signal statistics (used to initialise gamma and q) ────────────────────
    nry   = np.zeros(n)
    nry_q = np.zeros(n)
    Ttot  = 0.0
    for j in range(n_exp):
        ts_j   = data.ts[j]
        tsam_j = data.Tsam[j]
        dt     = tsam_j[1:] - tsam_j[:-1]
        diff2  = (ts_j[:, 1:] - ts_j[:, :-1]) ** 2
        nry   += (diff2 / dt).sum(axis=1)
        nry_q += diff2.sum(axis=1)
        Ttot  += tsam_j[-1] - tsam_j[0]
    nry   /= Ttot
    nry_q /= Ttot

    # ── Initial connectivity and hyperparameters ───────────────────────────────
    S    = np.random.rand(n, n + n_in) > 0.9
    bets = np.abs(np.random.randn(n, n + n_in) * 0.5) + 1e-3
    psi  = np.vstack([
        (maxs - mins)[:, None] * np.random.rand(n, nr_pi) + mins[:, None],
        np.random.rand(n_in, nr_pi)
    ])

    state = BINGOState(
        q     = nry_q / 20,
        gamma = nry,
        r     = np.full(n, 0.0006 * 0.1),
        xs    = xs,
        P     = np.full(n, -1e8),
        bets  = bets,
        J     = np.full(n, 1e8),
        S     = S,
        psi   = psi,
        ma    = np.full(n, 0.1),
        mb    = np.full(n, 0.05),
        Ser   = Ser,
    )

    return data, state


In [19]:
data, state = bingo_init(data, params=params, scale=True)

In [20]:
state.psi

array([[0.72550805, 1.329852  , 1.29316816, 0.73427881, 0.71392404, 1.2754268 , 0.80149907, 0.58959786, 1.13769893, 0.87221547, 1.25861006, 1.00606266, 0.65416322, 0.49248004, 0.88392721, 1.36449051, 0.85296556, 1.4166919 , 0.99580897, 0.7010209 , 0.50347705, 0.95116832, 1.19830027, 0.81490396, 1.40234493, 0.72591714, 1.13069741, 0.4926691 , 1.46954276, 1.14855029, 0.7671291 , 1.13207921, 0.79326511, 1.26340417, 0.64773546, 1.07921379, 1.18870733, 0.55055005, 1.38181174, 0.95404765, 0.9938341 , 1.44523223, 0.75180117, 1.12118416, 0.8978996 , 0.7328059 , 1.14798042, 0.62524293, 0.59029017, 0.7684956 ],
       [0.53067918, 0.88250078, 1.09179893, 0.42335536, 1.1445038 , 1.14203005, 0.79310911, 0.25439287, 1.0593882 , 0.58039551, 0.39446776, 0.26501084, 0.51114704, 0.82587431, 0.51376284, 0.52099636, 0.25859842, 0.70082772, 0.71946116, 0.8182258 , 0.46722731, 0.90009177, 0.34335049, 0.35183599, 0.67413918, 1.11340668, 0.95654213, 1.1196387 , 0.65638807, 0.56192201, 0.49011775, 0.56939568,

In [9]:
state.r.shape

(5,)

In [23]:
def bingo(data: BINGOData, state: BINGOState):
    """
    Main BINGO sampler.
    Returns: Plink, chain, xstore, state, stats
    """
    # ── Unpack state ───────────────────────────────────────────────────────────
    q       = state.q.copy()
    gamma   = state.gamma.copy()
    r       = state.r.copy()
    xs_old  = state.xs.copy()
    Pold    = state.P.copy()
    betsold = state.bets.copy()
    Jold    = state.J.copy()
    Sold    = state.S.copy()
    psiold  = state.psi.copy()
    ma      = state.ma.copy()
    mb      = state.mb.copy()
    Ser     = state.Ser  # [4 x n_experiments], 0-based

    # ── Basic dimensions ───────────────────────────────────────────────────────
    n     = data.n_genes
    n_in  = data.input[0].shape[0] if data.input is not None else 0
    nstep = data.params.nstep
    M     = psiold.shape[1]   # number of pseudo-inputs
    n_exp = data.n_experiments

    # ── Concatenate all time series into one matrix y ─────────────────────────
    y    = np.hstack(data.ts)                      # [n x total_timepoints]
    print(y.shape)
    Tsam = data.Tsam                                 # list of time vectors

    # ── Range of y (and inputs) for pseudo-input bounds ───────────────────────
    rany = np.vstack([y.min(axis=1), y.max(axis=1)]).T  # [n x 2]

    # ── All genes are included (no knockouts) ─────────────────────────────────
    geneList = np.arange(n)                          # 0-based

    # ── Log prior on links ────────────────────────────────────────────────────
    if data.params.link_pr is not None:
        log_link_pr = np.full((n, n + n_in), np.log(data.params.link_pr))
    else:
        log_link_pr = np.full((n, n + n_in), -np.log(n))

    # ── Prior network (sure links) ────────────────────────────────────────────
    S_aux = np.zeros((n, n + n_in))
    if data.sure is not None:
        S_aux[:n, :n] = data.sure
    Sold = np.maximum(Sold, S_aux)
    Sold = np.minimum(Sold, 1 + S_aux)
    S_aux = 1.0 - np.abs(S_aux)                     # free entries = 1

    # ── nomiss: all ones since we have no missing data ────────────────────────
    nomiss = np.ones((n, y.shape[1]))

    # ── derind_full, yind, d_full ──────────────────────────────────────────────
    # derind_full : indices into xs_old of the "left" point of each fine interval
    # yind        : indices into xs_old corresponding to measurement timepoints
    # d_full      : fine time step for each interval, repeated nstep times

    # first experiment
    n_tp0 = Ser[1, 0] - Ser[0, 0]                   # number of intervals in exp 0
    derind_full = np.arange(nstep * n_tp0) + Ser[2, 0]
    yind        = Ser[2, 0] + np.arange(Ser[1, 0] - Ser[0, 0] + 1) * nstep
    d_full      = np.repeat(np.diff(Tsam[0]) / nstep, nstep)

    for j in range(1, n_exp):
        n_tp_j      = Ser[1, j] - Ser[0, j]
        derind_j    = np.arange(nstep * n_tp_j) + Ser[2, j]
        yind_j      = Ser[2, j] + np.arange(n_tp_j + 1) * nstep
        d_j         = np.repeat(np.diff(Tsam[j]) / nstep, nstep)
        derind_full = np.concatenate([derind_full, derind_j])
        yind        = np.concatenate([yind, yind_j])
        d_full      = np.concatenate([d_full, d_j])

    # ── Signal statistics (quadratic variation, total variation) ──────────────
    nry        = np.zeros(n)
    totvar     = np.zeros(n)
    Total_time = np.zeros(n)
    for l in range(n_exp):
        ts_l   = data.ts[l]
        tsam_l = Tsam[l]
        dt_l   = tsam_l[1:] - tsam_l[:-1]
        diff_l = ts_l[:, 1:] - ts_l[:, :-1]
        nry       += (diff_l ** 2 / dt_l).sum(axis=1)
        totvar    += np.abs(diff_l).sum(axis=1)
        Total_time += tsam_l[-1] - tsam_l[0]
    nry    /= Total_time
    totvar /= Total_time

    # ── Piecewise linear embedding matrix Pr and interpolation matrix Pintc ───
    # Pr    : [mm x max_n_tp] maps measurement values to fine grid
    # Pintc : [nstep-1 x nstep-1] sine-based interpolation for CN sampler
    max_intervals = int((Ser[3, :] - Ser[2, :]).max()) + 1
    max_tp        = int((Ser[1, :] - Ser[0, :]).max()) + 1
    Pr = np.zeros((max_intervals, max_tp))
    t  = np.arange(1, nstep + 1) / nstep               # [1/nstep ... 1]
    Pr[:nstep, 0]    = t[::-1]                          # first column
    Pr[-nstep:, -1]  = t                                # last column
    for j in range(1, max_tp - 1):
        Pr[(j-1)*nstep+1 : j*nstep+1,     j] = t
        Pr[j*nstep       : (j+1)*nstep,   j] = t[::-1]

    k_vec  = np.arange(1, nstep).reshape(-1, 1)        # [nstep-1 x 1]
    Pintc  = (np.sin(k_vec * k_vec.T / nstep * np.pi)
              / (np.pi * k_vec.T) * 2 ** 0.5)          # [nstep-1 x nstep-1]

    # ── Initialise accumulators ────────────────────────────────────────────────
    Plink    = np.zeros_like(Sold)
    chain    = 0
    acctraj  = 0
    xstore   = np.zeros_like(xs_old)
    acctop   = np.zeros(n)
    acchyp   = np.zeros(n)
    accr     = np.zeros(n)
    yold     = xs_old[:, yind]

    
    for k in range(data.params.its):

        # ── Topology sampling ──────────────────────────────────────────────────
        for i in geneList:
            S = Sold[i, :].copy()

            # Decide whether to change topology or only hyperparameters
            top_change = float(np.random.rand() > 0.333)

            # Free entries for gene i (not forced by prior)
            inds = np.where(S_aux[i, :] > 0.5)[0]
            n_on = S[inds].sum()
            topc = float(
                np.random.rand() > 0.5
                and n_on > 0.5
                and n_on < len(inds) - 0.5
            )

            # Move type 1: flip one entry
            if top_change and not topc:
                indc = np.random.randint(len(inds))
                S[inds[indc]] = 1 - S[inds[indc]]

            # Move type 2: swap a 0 and a 1
            if top_change and topc:
                ind1 = np.where(S[inds] > 0.5)[0]
                ind0 = np.where(S[inds] < 0.5)[0]
                indc01 = ind0[np.random.randint(len(ind0))]
                indc10 = ind1[np.random.randint(len(ind1))]
                S[inds[indc01]] = 1
                S[inds[indc10]] = 0

            # Sample relevance parameters
            bets  = (1 - data.params.ebeta**2)**0.5 * betsold[i, :] + data.params.ebeta * np.random.randn(n + n_in)
            beta  = 0.5 + 0.45 * bets
            p_bets = np.exp(-np.abs(beta)) / np.exp(-(beta - 0.5)**2 / (2 * 0.45**2))
            beta  = np.abs(beta)

            # Sample other hyperparameters
            gamma_tr = gamma[i] + data.params.egamma * nry[i] * np.random.randn()
            gamma_tr = 1e-4 + abs(gamma_tr - 1e-4)
            matr = ma[i] + data.params.ea * np.random.randn()
            matr = 1e-7 + abs(matr - 1e-7)
            mbtr = mb[i] + data.params.eb * np.random.randn()
            mbtr = 1e-7 + abs(mbtr - 1e-7)

            # No knockouts: derind and d are always the full arrays
            derind = derind_full
            d      = d_full
            N      = len(derind)
            print(f'N is {N}')
            # Form covariance matrices
            KM  = np.zeros((M, M))
            KNM = np.zeros((N, M))
            for j in np.where(S[:n] > 0.5)[0]:
                diff_M  = psiold[j, :][None, :] - psiold[j, :][:, None]   # [M x M]
                diff_NM = psiold[j, :][None, :] - xs_old[j, derind][:, None]  # [N x M]
                KM  += beta[j] * diff_M**2
                KNM += beta[j] * diff_NM**2

            KM  = gamma_tr * np.exp(-KM)
            KNM = gamma_tr * np.exp(-KNM)
            print(f'sold is {Sold}')
            print(f'xs wanted { xs_old[0, derind[:5]]}')
            # Compute the load
            A   = KM + (1/q[i]) * (KNM.T * d) @ KNM + 1e-5 * np.eye(M)
            KC  = np.linalg.cholesky(A).T          # upper triangular, matches MATLAB chol
            der = ((xs_old[i, derind + 1] - xs_old[i, derind])
                   - d * (mbtr - matr * xs_old[i, derind])) / q[i]
            ld  = np.linalg.solve(KC, KNM.T @ der)
            print(f'km shape is {KM.shape}, KNM shape: {KNM.shape}\n KC shape = {KC.shape}')
            print(f'der shape {der.shape}')
            print(f'ld shape: {ld.shape}')
            print(f'km is {KM}')
            print(f'knm: {KNM}')
            print(f'ld is {ld}')
            # Wiener measure term (quadratic variation of yold)
            nrY = 0.0
            for l in range(n_exp):
                yi  = yold[i, Ser[0, l]:Ser[1, l] + 1]
                dt_l = Tsam[l][1:] - Tsam[l][:-1]
                nrY += ((yi[1:] - yi[:-1])**2 / dt_l).sum()

            # Cost function
            J1 = (0.5 * nrY / q[i]
                  - 0.5 * ld @ ld
                  + np.log(np.diag(KC)).sum()
                  - 0.5 * np.log(np.linalg.det(KM + 1e-5 * np.eye(M)))
                  - (mbtr - matr * xs_old[i, derind]) @ (xs_old[i, derind + 1] - xs_old[i, derind]) / q[i]
                  + 0.5 / q[i] * np.sum(d * (mbtr - matr * xs_old[i, derind])**2))

            PS = (S * log_link_pr[i, :]).sum() + np.log(p_bets).sum()

            # Acceptance
            P_aux_ab    = np.exp(0.1 * (ma[i] - matr + 2*mb[i] - 2*mbtr) / totvar[i])
            g_ratio     = (gamma_tr / nry[i] * (30 - gamma_tr / nry[i])
                           / (gamma[i] / nry[i] * (30 - gamma[i] / nry[i])))
            P_aux_gamma = g_ratio * np.exp(0.2 / nry[i] * (gamma[i] - gamma_tr))

            if P_aux_ab * P_aux_gamma * np.exp((PS - Pold[i] + Jold[i] - J1) / data.params.Theur) > np.random.rand():
                Sold[i, :]  = S
                Pold[i]     = PS
                Jold[i]     = J1
                betsold[i, :] = bets
                gamma[i]    = gamma_tr
                ma[i]       = matr
                mb[i]       = mbtr
                acctop[i]  += top_change
                acchyp[i]  += 1
            print(f'J1 is {J1}')
            print(f'ps is {PS}')
            # Sample measurement noise variance r[i]
            rtr = r[i] + data.params.er * np.random.randn()
            rtr = 1e-8 + abs(rtr - 1e-8)
            obs = np.where(nomiss[i, :] > 0.5)[0]
            log_acc_r = (
                (1 + len(obs) / 2) * np.log(r[i] / rtr)
                + 1e-5 / r[i] - 1e-5 / rtr
                + 0.5 * (1/r[i] - 1/rtr) * np.sum((y[i, obs] - yold[i, obs])**2)
            )
            if log_acc_r > np.log(np.random.rand()):
                r[i]      = rtr
                accr[i]  += 1

    # # ── Pack results ──────────────────────────────────────────────────────────
    # state.q     = q
    # state.gamma = gamma
    # state.r     = r
    # state.xs    = xs_old
    # state.P     = Pold
    # state.bets  = betsold
    # state.J     = Jold
    # state.S     = Sold
    # state.psi   = psiold
    # state.ma    = ma
    # state.mb    = mb

    # stats = {
    #     "acctraj": acctraj,
    #     "acctop":  acctop,
    #     "acchyp":  acchyp,
    #     "accr":    accr,
    # }
    return Ser, d_full, yind, Pr, Pintc

In [22]:
ser, d_full, yind, Pr, Pintc = bingo(data, state)

(5, 35)
N is 132
xs wanted [0.72056305 0.70814714 0.69573122 0.68331531 0.6708994 ]
km shape is (50, 50), KNM shape: (132, 50)
 KC shape = (50, 50)
der shape (132,)
ld shape: (50,)
km is [[0.05323727 0.03629478 0.04213866 0.03463468 0.04825609 0.04091729 0.03516789 0.05224289 0.04232226 0.05014702 0.0416475  0.0463674  0.05243542 0.03739181 0.05011725 0.03948165 0.05088969 0.03556254 0.04065967 0.05274395 0.05059746 0.05091668 0.04235835 0.05292359 0.0380169  0.03626505 0.03879104 0.040403   0.03238782 0.0467838  0.04116126 0.04584382 0.04851231 0.04320636 0.05297173 0.03872417 0.04543974 0.04840499 0.03613179 0.03974391 0.03527684 0.03658785 0.04134534 0.03551862 0.03533172 0.05314898 0.04641367 0.03714652 0.0525072  0.05108935]
 [0.03629478 0.05323727 0.04848685 0.03748379 0.04046874 0.05218784 0.03993748 0.03332241 0.05182144 0.04494553 0.05194939 0.04918276 0.03619045 0.03069764 0.04522965 0.04928801 0.04399975 0.05239426 0.04843553 0.0329746  0.02609328 0.04493036 0.05218624 0.038

/tmp/ipykernel_29834/1758074358.py:216: RuntimeWarning: overflow encountered in exp
  if P_aux_ab * P_aux_gamma * np.exp((PS - Pold[i] + Jold[i] - J1) / data.params.Theur) > np.random.rand():


In [17]:
print(f'ser: {ser}, d_full:{d_full}, \n yind:{yind} \n Pr={Pr}: Pintc: {Pintc}')

ser: [[  0  21]
 [ 20  34]
 [  0  81]
 [ 80 133]], d_full:[0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125
 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125
 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125
 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125
 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125
 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125
 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.25  0.25  0.25  0.25
 0.15  0.15  0.15  0.15  0.15  0.15  0.15  0.15  0.2   0.2   0.2   0.2
 0.25  0.25  0.25  0.25  0.225 0.225 0.225 0.225 0.075 0.075 0.075 0.075
 0.2   0.2   0.2   0.2   0.275 0.275 0.275 0.275 0.2   0.2   0.2   0.2
 0.15  0.15  0.15  0.15  0.15  0.15  0.15  0.15  0.225 0.225 0.225 0.225], 
 yind:[  0   4   8  12  16  20  24  28  32  36  40  44  48  52  56  60  64  68
  72  76  80  81  85  89  93  97 101 105 109 113 117 121 125 1

In [19]:
print(Pr)

[[1.   0.   0.   ... 0.   0.   0.  ]
 [0.75 0.25 0.   ... 0.   0.   0.  ]
 [0.5  0.5  0.   ... 0.   0.   0.  ]
 ...
 [0.   0.   0.   ... 0.   0.5  0.5 ]
 [0.   0.   0.   ... 0.   0.25 0.75]
 [0.   0.   0.   ... 0.   0.   1.  ]]
